In [1]:
import os 
import networkx as nx
from scipy.special import kl_div
from collections import Counter
import numpy as np
import torch
import itertools 

In [2]:
def get_node_dist(node_types):
    dist_dict = Counter(node_types)
    dist_dict = dict(sorted(dist_dict.items()))
    return list(dist_dict.values())   

Load original graphs

In [3]:
file_path = '../../real_dblp/15.pt'

In [4]:
adjs, types = torch.load(file_path)

In [5]:
path = 'dblp_syn_small_15/' 
files = os.listdir(path)

In [6]:
#len(files)

In [7]:
node_types_list = []

In [8]:
for node_type in types:
    if set([0,1,2,3]).issubset(node_type):
        node_types_list.append(node_type)

In [9]:
#len(node_types_list)

In [10]:
def get_marginal_prob(node_types):
    
    if set([0,1,2,3]).issubset(node_types):
        node_dist= get_node_dist(node_types)
        marginal_prob = np.array(node_dist) / np.array(node_dist).sum()
        return marginal_prob

## Diffusion

In [11]:
diff_kl_div_list = []

In [12]:
counter = 0
for index, file in enumerate(files):
    #print(index)
    
    if file.endswith('.gexf'):

        filepath = os.path.join(path, file)
        G_syn_diff= nx.read_gexf(filepath)
        node_types_diff=nx.get_node_attributes(G_syn_diff, "node_type")

        node_types_syn = list(node_types_diff.values())
        if set([0,1,2,3]).issubset(node_types_syn):
            #print('a')
            #print(node_types_list[index])
            if counter<len(node_types_list)-1:
                real_marginal_prob = get_marginal_prob(node_types_list[counter])
                #print('real_marginal_prob',real_marginal_prob)
                #print(node_types_syn)
    
                sync_marginal_prob_diff =  get_marginal_prob(node_types_syn)
                #print(sync_marginal_prob_diff)
                kl_divergence_diff = kl_div(real_marginal_prob, sync_marginal_prob_diff).sum()
                #print('kl_div',kl_divergence_diff)
                diff_kl_div_list.append(kl_divergence_diff)
                counter = counter+1

In [13]:
len(diff_kl_div_list[:50])

50

In [14]:
np.mean(diff_kl_div_list[:50])

0.13884966411527774

## VAE

In [15]:
file_path_vae = '../../real_dblp_vae/15.pt'

In [16]:
adjs_vae, types_vae, node_feats = torch.load(file_path_vae)

In [17]:
#types_vae

In [18]:
vae_node_types_list = []

In [19]:
for node_type in types_vae:
    if set([0,1,2,3]).issubset(node_type):
        vae_node_types_list.append(node_type)

In [20]:
#len(vae_node_types_list)

In [21]:
#vae_node_types_list

In [22]:
vae_kl_div_list = []

In [23]:
rootdir = 'dblp_vae_15'
dir_list = []
counter = 0
for subdir, dirs,files in os.walk(rootdir):
   
    if files:
        graph_path = os.path.join(subdir, files[0]) 
        node_type_path = os.path.join(subdir, files[1]) 
        node_types_syn_vae = torch.load(node_type_path).detach().numpy()
        
        #print(node_types_vae)
        if set([0,1,2,3]).issubset(node_types_syn_vae):
            
            #print(node_types_syn_vae)
            #node_dist_syn_vae= get_node_dist(node_types_syn_vae)
            if counter<len(vae_node_types_list)-1:
                real_marginal_prob_vae = get_marginal_prob(vae_node_types_list[counter])
                #print(counter)
                #print(vae_node_types_list[index])
    
                syn_marginal_prob_vae =  get_marginal_prob(node_types_syn_vae)
    
                kl_divergence_vae = kl_div(real_marginal_prob_vae, syn_marginal_prob_vae).sum()
                
                #syn_marginal_prob_vae = np.array(node_dist_syn_vae) / np.array(node_dist_syn_vae).sum()       
                #kl_divergence_vae = kl_div(real_marginal_prob_vae, syn_marginal_prob_vae).sum()
                vae_kl_div_list.append(kl_divergence_vae)
                counter = counter+1

In [24]:
len(vae_kl_div_list[:50])

50

In [25]:
np.mean(vae_kl_div_list[:50])

0.37583932799881814

In [26]:
#len(vae_kl_div_list)